In [1]:
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import re

In [2]:
# Download NLTK resources
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package punkt to
[nltk_data]     /Users/user/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/user/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/user/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [4]:
# Load the Kindle reviews dataset (replace with actual file path)
# Note: Students should download a sample dataset, e.g., from Kaggle
data = pd.read_csv('kindle_reviews.csv')

In [5]:
# Preprocess text
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

In [6]:
def preprocess_text(text):
    # Handle non-string inputs
    if not isinstance(text, str):
        return ""
    # Convert to lowercase and remove punctuation
    text = re.sub(r'[^\w\s]', '', text.lower())
    # Tokenize
    words = word_tokenize(text)
    # Remove stop words and lemmatize
    words = [lemmatizer.lemmatize(word) for word in words if word not in stop_words]
    return ' '.join(words)

In [7]:
# Apply preprocessing
data['cleaned_review'] = data['reviewText'].apply(preprocess_text)

# Create sentiment labels: 1-2 stars = negative (0), 4-5 stars = positive (1)
data['sentiment'] = data['overall'].apply(lambda x: 0 if x <= 2 else 1 if x >= 4 else -1)
data = data[data['sentiment'] != -1]  # Remove neutral reviews

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    data['cleaned_review'], data['sentiment'], test_size=0.2, random_state=42
)

# Convert text to TF-IDF features
vectorizer = TfidfVectorizer(max_features=5000)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

In [8]:
# Train the logistic regression model
model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)

# Make predictions and evaluate
predictions = model.predict(X_test_tfidf)
print("Accuracy:", accuracy_score(y_test, predictions))
print("Detailed Report:\n", classification_report(y_test, predictions))

# Optional: Display important words
feature_names = vectorizer.get_feature_names_out()
coefficients = model.coef_[0]
word_importance = pd.DataFrame({'word': feature_names, 'coefficient': coefficients})
print("\nTop 5 Positive Words:")
print(word_importance.sort_values(by='coefficient', ascending=False).head())
print("\nTop 5 Negative Words:")
print(word_importance.sort_values(by='coefficient').head())

/opt/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: divide by zero encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/opt/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: overflow encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)
/opt/anaconda3/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:209: RuntimeWarning: invalid value encountered in matmul
  norm2_w = weights @ weights if weights.ndim == 1 else squared_norm(weights)


Accuracy: 0.9670022844572299
Detailed Report:
               precision    recall  f1-score   support

           0       0.84      0.61      0.71     11588
           1       0.97      0.99      0.98    165697

    accuracy                           0.97    177285
   macro avg       0.91      0.80      0.84    177285
weighted avg       0.96      0.97      0.96    177285


Top 5 Positive Words:
           word  coefficient
2730      loved    12.595130
1500    enjoyed    12.102084
4794       wait    11.270788
1976      great     9.203306
1581  excellent     8.985766

Top 5 Negative Words:
               word  coefficient
4823          waste    -8.975814
3326         poorly    -8.033060
1174        deleted    -7.507797
4687  unfortunately    -7.481628
1286  disappointing    -7.115146
